# MoE Active and Inactive Experts

This notebook defines the MoE compression ablation. The default run is a deterministic CPU toy MoE with active and inactive experts represented as separate tensors. A real Hugging Face MoE sweep is prepared behind a CUDA guard and is not executed by default.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "outputs"

import pandas as pd
import torch

from experiment_utils import collect_exponents, global_huffman_exponent_bits, neuralzip_v2_exponent_bits
from model_compression import HuggingFaceModel
from neuralzip_model import NeuralZipModel

try:
    from IPython.display import display
except ImportError:
    display = print

/Users/martin/miniconda3/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.manual_seed(41)
ACTIVE_EXPERTS = {0, 2}
expert_tensors = []

for expert_id in range(4):
    is_active = expert_id in ACTIVE_EXPERTS
    scale = 1.0 if is_active else 0.03
    expert_tensors.extend(
        [
            (f"moe.experts.{expert_id}.w1.weight", torch.randn(64, 64, dtype=torch.float32) * scale),
            (f"moe.experts.{expert_id}.w2.weight", torch.randn(64, 64, dtype=torch.float32) * (scale / 2)),
        ]
    )

expert_rows = []
for expert_id in range(4):
    tensors = [(name, tensor) for name, tensor in expert_tensors if f"experts.{expert_id}." in name]
    expert_model = NeuralZipModel(tensors)
    named_exponents = expert_model.named_exponents()
    layer_exponents = [exps for _, exps in named_exponents]
    raw_bits = sum(exps.size for exps in layer_exponents) * 8
    v2_bits, v2_stats = neuralzip_v2_exponent_bits(expert_model, named_exponents)
    expert_rows.append(
        {
            "expert": expert_id,
            "routing_group": "active" if expert_id in ACTIVE_EXPERTS else "inactive",
            "raw_exponent_bits": int(raw_bits),
            "global_huffman_exponent_bits": int(global_huffman_exponent_bits(expert_model, layer_exponents)),
            "neuralzip_v2_exponent_bits": int(v2_bits),
            "neuralzip_v2_clusters": int(v2_stats["clusters"]),
        }
    )

expert_table = pd.DataFrame(expert_rows)
for column in ["global_huffman_exponent_bits", "neuralzip_v2_exponent_bits"]:
    expert_table[f"{column}_ratio_vs_raw"] = expert_table[column] / expert_table["raw_exponent_bits"]

display(expert_table)

,expert,routing_group,raw_exponent_bits,global_huffman_exponent_bits,neuralzip_v2_exponent_bits,neuralzip_v2_clusters,global_huffman_exponent_bits_ratio_vs_raw,neuralzip_v2_exponent_bits_ratio_vs_raw
0,0,active,65536,22992,22840,2,0.350830,0.348511
1,1,inactive,65536,22576,22216,2,0.344482,0.338989
2,2,active,65536,22800,22632,2,0.347900,0.345337
3,3,inactive,65536,22776,22568,2,0.347534,0.344360


In [3]:
moe_model = NeuralZipModel(expert_tensors)
moe_named_exponents = moe_model.named_exponents()
moe_layer_exponents = [exps for _, exps in moe_named_exponents]
moe_v2_bits, moe_v2_stats = neuralzip_v2_exponent_bits(moe_model, moe_named_exponents)

moe_summary = pd.DataFrame([
    {
        "scope": "all experts together",
        "raw_exponent_bits": int(sum(exps.size for exps in moe_layer_exponents) * 8),
        "global_huffman_exponent_bits": int(global_huffman_exponent_bits(moe_model, moe_layer_exponents)),
        "neuralzip_v2_exponent_bits": int(moe_v2_bits),
        "neuralzip_v2_clusters": int(moe_v2_stats["clusters"]),
    }
])
display(moe_summary)
moe_v2_stats

,scope,raw_exponent_bits,global_huffman_exponent_bits,neuralzip_v2_exponent_bits,neuralzip_v2_clusters
0,all experts together,262144,114968,89864,8


{'clusters': 8, 'families': 2, 'orders': {1: 8}}

In [4]:
RUN_REAL_MOE_GPU_SWEEP = False
MOE_MODEL_ID = "mistralai/Mixtral-8x7B-v0.1"

if RUN_REAL_MOE_GPU_SWEEP:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for the real MoE sweep; synthetic CPU cells above are safe to run.")

    moe = HuggingFaceModel(MOE_MODEL_ID, dtype="bfloat16")
    expert_named_exponents = [
        (name, exps)
        for name, exps in moe.named_exponents()
        if "expert" in name.lower() or ".experts." in name.lower()
    ]
    expert_layer_exponents = [exps for _, exps in expert_named_exponents]
    display(pd.DataFrame([
        {
            "model_id": MOE_MODEL_ID,
            "expert_tensor_count": len(expert_named_exponents),
            "raw_exponent_bits": int(sum(exps.size for exps in expert_layer_exponents) * 8),
            "global_huffman_exponent_bits": int(global_huffman_exponent_bits(moe, expert_layer_exponents)),
        }
    ]))